<a href="https://colab.research.google.com/github/gmauricio-toledo/tda-gdl/blob/main/09-Homolog%C3%ADa_Persistente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Homología Persistente</h1>

En esta notebook experimentaremos con la Homología Persistente. Primero ilustraremos las coordenadas baricéntricas para definir puntos dentro de simplejos. Posteriormente, construimos los complejos de Cech y Vietoris-Rips.

# Simplejos y coordenadas baricéntricas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Definición de los puntos
a0 = np.array([0, 0])
a1 = np.array([1, -0.5])
a2 = np.array([0.3, 1])
puntos = np.array([a0, a1, a2])

def graficar_punto(t0, t1):
    t2 = 1 - t0 - t1
    punto = t0 * a0 + t1 * a1 + t2 * a2

    plt.figure(figsize=(8, 6))
    plt.scatter(puntos[:, 0], puntos[:, 1], c='blue', label='Vértices')
    if t2>=0:
        plt.scatter(punto[0], punto[1], c='green', label='Punto')
    else:
        plt.scatter(punto[0], punto[1], c='red', label='Punto')
    plt.fill(puntos[:, 0], puntos[:, 1], alpha=0.3, color='lightblue')  # Rellenar el triángulo
    plt.plot([puntos[0, 0], puntos[1, 0], puntos[2, 0], puntos[0, 0]],
             [puntos[0, 1], puntos[1, 1], puntos[2, 1], puntos[0, 1]], 'b-')  # Dibujar el triángulo
    plt.title(f't0={t0:.2f}, t1={t1:.2f}, t2={t2:.2f}')
    plt.legend()
    plt.axis('equal')
    plt.show()

# Sliders para t0 y t1
interact(
    graficar_punto,
    t0=FloatSlider(min=0, max=1, step=0.01, value=0.3, description='t0:'),
    t1=FloatSlider(min=0, max=1, step=0.01, value=0.4, description='t1:')
)

# Complejos Simpliciales

In [ ]:
!pip install gudhi

Ejemplo 1: Circulo con ruido

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ejemplo 1: Círculo ruidoso
theta = np.linspace(0, 2*np.pi, 30)
X = np.column_stack([np.cos(theta), np.sin(theta)])
X += np.random.normal(0, 0.1, X.shape)
print(X.shape)

plt.figure()
plt.scatter(X[:, 0], X[:, 1])
plt.show()

In [ ]:
import gudhi
import numpy as np
import matplotlib.pyplot as plt

# Dos círculos concéntricos
np.random.seed(42)
n_inner, n_outer = 30, 40

theta_inner = np.linspace(0, 2*np.pi, n_inner, endpoint=False)
X_inner = 0.5 * np.column_stack([np.cos(theta_inner), np.sin(theta_inner)])
X_inner += np.random.normal(0, 0.05, X_inner.shape)

theta_outer = np.linspace(0, 2*np.pi, n_outer, endpoint=False)
X_outer = 1.5 * np.column_stack([np.cos(theta_outer), np.sin(theta_outer)])
X_outer += np.random.normal(0, 0.05, X_outer.shape)

X = np.vstack([X_inner, X_outer])

print(X.shape)

plt.figure()
plt.scatter(X[:, 0], X[:, 1])
plt.axis('equal')
plt.axis('off')
plt.show()

## Complejos Simpliciales: Cech y Vietoris-Rips

In [ ]:
# Construir y visualizar para diferentes epsilons
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
epsilons = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]

for ax, eps in zip(axes.flat, epsilons):
    rips = gudhi.RipsComplex(points=X, max_edge_length=eps)
    st = rips.create_simplex_tree(max_dimension=2)

    ax.scatter(X[:, 0], X[:, 1], c='black', s=30, zorder=3, alpha=0.6)

    # Aristas
    for simplex in st.get_simplices():
        if len(simplex[0]) == 2:
            i, j = simplex[0]
            ax.plot([X[i,0], X[j,0]], [X[i,1], X[j,1]],
                   'b-', alpha=0.2, linewidth=0.8)

    # Contar componentes y ciclos
    n_edges = sum(1 for s in st.get_simplices() if len(s[0]) == 2)
    n_triangles = sum(1 for s in st.get_simplices() if len(s[0]) == 3)

    # Rellenar triángulos (2-simplejos)
    for simplex in st.get_simplices():
        if len(simplex[0]) == 3:
            i, j, k = simplex[0]
            triangle = plt.Polygon(X[[i,j,k]], alpha=0.15, color='lightblue')
            ax.add_patch(triangle)

    ax.set_title(f'ε={eps}: {n_edges} aristas, {n_triangles} triángulos')
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from ipywidgets import interact, FloatSlider
import matplotlib.pyplot as plt
import gudhi

def visualizar_rips(epsilon):
    """Visualiza el complejo de Rips para un valor de epsilon dado"""
    fig, ax = plt.subplots()

    # Crear el complejo de Rips
    rips = gudhi.RipsComplex(points=X, max_edge_length=epsilon)
    st = rips.create_simplex_tree(max_dimension=2)

    # Dibujar puntos
    ax.scatter(X[:, 0], X[:, 1], c='black', s=50, zorder=3, alpha=0.7)

    # Dibujar aristas
    for simplex in st.get_simplices():
        if len(simplex[0]) == 2:
            i, j = simplex[0]
            ax.plot([X[i,0], X[j,0]], [X[i,1], X[j,1]],
                   'b-', alpha=0.3, linewidth=1.5)

    # Rellenar triángulos (2-simplejos)
    for simplex in st.get_simplices():
        if len(simplex[0]) == 3:
            i, j, k = simplex[0]
            triangle = plt.Polygon(X[[i,j,k]], alpha=0.2, color='lightblue')
            ax.add_patch(triangle)

    # Contar elementos
    n_edges = sum(1 for s in st.get_simplices() if len(s[0]) == 2)
    n_triangles = sum(1 for s in st.get_simplices() if len(s[0]) == 3)

    ax.set_title(f'Complejo de Rips: ε={epsilon:.2f}\n{n_edges} aristas, {n_triangles} triángulos',
                 fontsize=14)
    ax.set_aspect('equal')
    ax.axis('off')

    plt.tight_layout()
    plt.show()

# Crear el slider interactivo
interact(visualizar_rips,
         epsilon=FloatSlider(value=0.6, min=0.1, max=2, step=0.05,
                            description='Epsilon:', continuous_update=False))

Podemos contar los simplejos en cada dimensión

In [ ]:
rips = gudhi.RipsComplex(points=X, max_edge_length=1)
simplex_tree = rips.create_simplex_tree(max_dimension=3)

for dim in range(4):
    print(f"{dim}-simplejos: {sum(1 for s in simplex_tree.get_simplices() if len(s[0])-1 == dim)}")